# Multi-Task FM-MLP: Dataset & Model Architecture

In [ ]:
import json
import os

import torch
import torch.nn as nn

from kkbox import config as kkbox_config
from kkbox.data import columns_from_manifest, load_splits, make_loader, stratified_subsample
from kkbox.determinism import seed_everything
from kkbox.models import FMInteractionLayer, MultiTaskFMNet, build_model

CFG = kkbox_config.load_config()
PROCESSED_DIR = kkbox_config.abspath(CFG, CFG["paths"]["processed_dir"])
# Set via env var for `make smoke` (2% stratified subsample); unset for a real run.
SUBSAMPLE_FRAC = float(os.environ["KKBOX_SUBSAMPLE_FRAC"]) if "KKBOX_SUBSAMPLE_FRAC" in os.environ else None

seed_everything(CFG["seeds"]["default"])

with open(os.path.join(PROCESSED_DIR, "feature_manifest.json")) as f:
    manifest = json.load(f)

CAT_COLS, NUM_COLS, CARDINALITIES, EMBED_DIMS = columns_from_manifest(manifest)

print(f"categorical columns ({len(CAT_COLS)}): {CAT_COLS}")
print(f"numerical columns ({len(NUM_COLS)}): {NUM_COLS}")
print(f"cardinalities: {CARDINALITIES}")
print(f"embedding dims: {EMBED_DIMS}")

## PyTorch `Dataset` & `DataLoader`

`CARDINALITIES` already includes the reserved "unknown" bucket from `02_Feature_Engineering.ipynb`'s label encoding (e.g. `city_enc` cardinality is 22, with index 21 reserved for missing/unseen cities), so `nn.Embedding(num_embeddings=cardinality, ...)` is used directly below — no extra `+1` is needed on top of that.

In [ ]:
splits = load_splits(PROCESSED_DIR)
train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

if SUBSAMPLE_FRAC is not None:
    train_df = stratified_subsample(train_df, SUBSAMPLE_FRAC)
    val_df = stratified_subsample(val_df, SUBSAMPLE_FRAC)
    test_df = stratified_subsample(test_df, SUBSAMPLE_FRAC)
    print(f"SUBSAMPLE_FRAC={SUBSAMPLE_FRAC} applied (stratified by is_churn)")

train_loader = make_loader(train_df, CAT_COLS, NUM_COLS, CFG["training"]["batch_size"],
                            shuffle=True, seed=CFG["seeds"]["default"])
val_loader = make_loader(val_df, CAT_COLS, NUM_COLS, CFG["training"]["batch_size"], shuffle=False)
test_loader = make_loader(test_df, CAT_COLS, NUM_COLS, CFG["training"]["batch_size"], shuffle=False)

print(f"train: {len(train_df):,} rows / {len(train_loader)} batches")
print(f"val:   {len(val_df):,} rows / {len(val_loader)} batches")
print(f"test:  {len(test_df):,} rows / {len(test_loader)} batches")

## 3.4 Factorization Machine Interaction Layer

Rendle, S. (2010), *Factorization Machines*, IEEE ICDM. Computes all pairwise feature interactions in O(k·d) instead of O(d²) via the sum-of-squares-minus-square-of-sums identity.

Implementation: `kkbox.models.FMInteractionLayer` (imported above) -

```python
class FMInteractionLayer(nn.Module):
    def __init__(self, input_dim, k=8):
        super().__init__()
        self.V = nn.Parameter(torch.randn(input_dim, k) * 0.01)

    def forward(self, x):  # x shape: (batch, input_dim)
        xV = x.unsqueeze(2) * self.V.unsqueeze(0)  # (batch, input_dim, k)
        sum_then_sq = xV.sum(dim=1).pow(2)  # (batch, k)
        sq_then_sum = xV.pow(2).sum(dim=1)  # (batch, k)
        return 0.5 * (sum_then_sq - sq_then_sum)  # (batch, k)
```

## 4.1 Full Model Architecture (FM + MLP + Dual Heads)

Embeddings + numerical features are concatenated (27 dims), passed through the FM layer (8-dim interaction vector), concatenated again (35 dims), then through the shared MLP backbone (Dense+BatchNorm+ReLU+Dropout × 3, ending at a 64-dim shared representation) into two independent heads.

Implementation: `kkbox.models.MultiTaskFMNet` (imported above), built via `kkbox.models.build_model(cardinalities, embed_dims, cat_cols, num_numerical, model_cfg)` which reads `fm_k`/`backbone_dims`/`dropouts`/`head_hidden_dim` from `config.yaml`'s `model` section rather than hard-coding them.

## Instantiate and inspect the architecture

In [3]:
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


model = build_model(CARDINALITIES, EMBED_DIMS, CAT_COLS, len(NUM_COLS), CFG["model"])
print(model)

combined_dim = sum(EMBED_DIMS.values()) + len(NUM_COLS)
print(f"\ncombined pre-FM input dim: {combined_dim} (plan: 27)")
print(f"backbone input dim (post-FM): {combined_dim + CFG['model']['fm_k']} (plan: 35)")

print("\nparameter counts:")
for name, part in [
    ("embeddings", model.embeddings),
    ("fm", model.fm),
    ("backbone", model.backbone),
    ("churn_head", model.churn_head),
    ("ltv_head", model.ltv_head),
]:
    print(f"  {name:12s} {count_params(part):>8,}")
print(f"  {'total':12s} {count_params(model):>8,}")

MultiTaskFMNet(
  (embeddings): ModuleDict(
    (city_enc): Embedding(22, 5)
    (gender_enc): Embedding(4, 2)
    (registered_via_enc): Embedding(7, 3)
    (payment_method_id_enc): Embedding(40, 8)
  )
  (fm): FMInteractionLayer()
  (backbone): Sequential(
    (0): Linear(in_features=35, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
  )
  (churn_head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=Tru

## Forward + backward pass sanity check

Pull one real batch, run it through the model, compute both losses (`BCEWithLogitsLoss` with `pos_weight` derived from the actual training-set class imbalance, and `MSELoss` on `log1p_ltv`), and confirm gradients flow back through the whole network. The loss values themselves are meaningless here since the model is randomly initialized — this only confirms the architecture and loss wiring are correct before any training happens.

In [4]:
x_num, x_cat, y_churn, y_ltv = next(iter(train_loader))
print(f"batch shapes: x_num={tuple(x_num.shape)}, x_cat={tuple(x_cat.shape)}, "
      f"y_churn={tuple(y_churn.shape)}, y_ltv={tuple(y_ltv.shape)}")

model.train()
churn_logit, ltv_pred = model(x_num, x_cat)
print(f"forward output shapes: churn_logit={tuple(churn_logit.shape)}, ltv_pred={tuple(ltv_pred.shape)}")

pos_weight = torch.tensor((train_df["is_churn"] == 0).sum() / (train_df["is_churn"] == 1).sum())
print(f"pos_weight (from actual train churn rate): {pos_weight.item():.2f} (plan estimate: ~12-14)")

bce_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
mse_loss_fn = nn.MSELoss()
loss_churn = bce_loss_fn(churn_logit, y_churn)
loss_ltv = mse_loss_fn(ltv_pred, y_ltv)
print(f"sanity-check losses (untrained model): bce={loss_churn.item():.4f}, mse={loss_ltv.item():.4f}")

(loss_churn + loss_ltv).backward()
all_grads_populated = all(p.grad is not None for p in model.parameters() if p.requires_grad)
print(f"backward() OK, all parameters have gradients: {all_grads_populated}")

batch shapes: x_num=(2048, 9), x_cat=(2048, 4), y_churn=(2048,), y_ltv=(2048,)
forward output shapes: churn_logit=(2048,), ltv_pred=(2048,)
pos_weight (from actual train churn rate): 0.36 (plan estimate: ~12-14)
sanity-check losses (untrained model): bce=0.3682, mse=15.4759
backward() OK, all parameters have gradients: True
